# Lab4: Cleaning & Preparing Data
## 65070501050 | Wongsatorn Sungsilapawech

**NOTE**: The summary of the techniques used are summarized in the bottom of this notebook.

# Web Scraping

In [1]:
from bs4 import BeautifulSoup, NavigableString
import requests

> I chose the thread `What is the differences between builder design pattern and factory design pattern?`

In [2]:
url = 'https://stackoverflow.com/questions/757743/what-is-the-difference-between-builder-design-pattern-and-factory-design-pattern'

In [3]:
page = requests.get(url)
soup = BeautifulSoup(page.text, 'html')

> In StackOverflow, the tag containing the thread's question is labeled with class name: `question-hyperlink`
```html
<a href="link-to-question" class="question-hyperlink">
```

In [17]:
question_text = soup.find('a',class_='question-hyperlink').text

In [18]:
question_text

'What is the difference between Builder Design pattern and Factory Design pattern? [closed]'

> In StackOverflow, the tag that contains the thread's answer is labeled with class name: `answercell post-layout--right`
>
```html
<div class="answercell post-layout--right">
```

In [4]:
all_ans = soup.find_all('div', class_='answercell post-layout--right')

In [5]:
ans_text = [ans.find_all('div', class_='s-prose js-post-body')[0] for ans in all_ans]

In [6]:
len(ans_text)

27

> The number of answers in this thread: $27$

Look for some portions of answers

In [7]:
for i in range(2):
    print(repr(ans_text[i].text))
    print('=========================================')

'\nWith design patterns, there usually is no "more advantageous" solution that works for all cases. It depends on what you need to implement.\nFrom Wikipedia: \n\n\nBuilder focuses on constructing a\n  complex object step by step. Abstract\n  Factory emphasizes a family of product\n  objects (either simple or complex).\n  Builder returns the product as a final\n  step, but as far as the Abstract\n  Factory is concerned, the product gets\n  returned immediately.\nBuilder often builds a Composite.\nOften, designs start out using Factory Method (less complicated, more\n  customizable, subclasses proliferate)\n  and evolve toward Abstract Factory,\n  Prototype, or Builder (more flexible,\n  more complex) as the designer\n  discovers where more flexibility is\n  needed.\nSometimes creational patterns are complementary: Builder can use one\n  of the other patterns to implement\n  which components get built. Abstract\n  Factory, Builder, and Prototype can\n  use Singleton in their\n  implemen

## Extract the `Code` environment answer out from the answer text

In [8]:
ans_code = [[] for i in range(len(ans_text))]

In [9]:
len(ans_code)

27

In [10]:
for i, ans in enumerate(ans_text):
    code = ans.find_all('code')
    for j in range(len(code)):
        ans_code[i].append(code[j].text)

In [11]:
print(ans_code[1])

['// Factory\nstatic class FruitFactory {\n    static Fruit create(name, color, firmness) {\n        // Additional logic\n        return new Fruit(name, color, firmness);\n    }\n}\n\n// Usage\nFruit fruit = FruitFactory.create("apple", "red", "crunchy");\n', '// Builder\nclass FruitBuilder {\n    String name, color, firmness;\n    FruitBuilder setName(name)         { this.name     = name;     return this; }\n    FruitBuilder setColor(color)       { this.color    = color;    return this; }\n    FruitBuilder setFirmness(firmness) { this.firmness = firmness; return this; }\n    Fruit build() {\n        return new Fruit(this); // Pass in the builder\n    }\n}\n\n// Usage\nFruit fruit = new FruitBuilder()\n        .setName("apple")\n        .setColor("red")\n        .setFirmness("crunchy")\n        .build();\n']


## Remove the `Code` environment from the answer to get the `pure text answer` (by performing **DATA MASKING** the code block as `[CODE]`)

In [12]:
for i, ans in enumerate(ans_text):
    for code in ans.find_all('code'):
        placeholder = NavigableString('[CODE]')
        code.replace_with(placeholder)

In [16]:
print(ans_text[1].text)


A factory is simply a wrapper function around a constructor (possibly one in a different class). The key difference is that a factory method pattern requires the entire object to be built in a single method call, with all the parameters passed in on a single line. The final object will be returned.
A builder pattern, on the other hand, is in essence a wrapper object around all the possible parameters you might want to pass into a constructor invocation. This allows you to use setter methods to slowly build up your parameter list. One additional method on a builder class is a build() method, which simply passes the builder object into the desired constructor, and returns the result.
In static languages like Java, this becomes more important when you have more than a handful of (potentially optional) parameters, as it avoids the requirement to have telescopic constructors for all the possible combinations of parameters. Also a builder allows you to use setter methods to define read-only

> You can see that the code environments are all replaced with `[CODE]` mask
> 
> And the real code is in the another variable

In [26]:
for i in range(len(ans_code[1])):
    print(ans_code[1][i])
    print('=======================================================================')

// Factory
static class FruitFactory {
    static Fruit create(name, color, firmness) {
        // Additional logic
        return new Fruit(name, color, firmness);
    }
}

// Usage
Fruit fruit = FruitFactory.create("apple", "red", "crunchy");

// Builder
class FruitBuilder {
    String name, color, firmness;
    FruitBuilder setName(name)         { this.name     = name;     return this; }
    FruitBuilder setColor(color)       { this.color    = color;    return this; }
    FruitBuilder setFirmness(firmness) { this.firmness = firmness; return this; }
    Fruit build() {
        return new Fruit(this); // Pass in the builder
    }
}

// Usage
Fruit fruit = new FruitBuilder()
        .setName("apple")
        .setColor("red")
        .setFirmness("crunchy")
        .build();



# Perform Data Cleansing and Data Preprocessing

In [27]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK resources (if not already downloaded)
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [28]:
question_text

'What is the difference between Builder Design pattern and Factory Design pattern? [closed]'

## Perform **Stopwords filter**, togethered with **Word Tokenization**

In [31]:
stop_words = set(stopwords.words('english'))

### Question filter

In [59]:
question_token = word_tokenize(question_text)

In [60]:
question_filtered = [word for word in question_token if word.lower() not in stop_words]

In [61]:
print(question_filtered)

['difference', 'Builder', 'Design', 'pattern', 'Factory', 'Design', 'pattern', '?', '[', 'closed', ']']


### Answer filter

In [43]:
ans_token = []

In [46]:
for i, ans in enumerate(ans_text):
    token = word_tokenize(ans_text[i].text)
    ans_token.append(token)    

In [53]:
print(ans_token[1])

['A', 'factory', 'is', 'simply', 'a', 'wrapper', 'function', 'around', 'a', 'constructor', '(', 'possibly', 'one', 'in', 'a', 'different', 'class', ')', '.', 'The', 'key', 'difference', 'is', 'that', 'a', 'factory', 'method', 'pattern', 'requires', 'the', 'entire', 'object', 'to', 'be', 'built', 'in', 'a', 'single', 'method', 'call', ',', 'with', 'all', 'the', 'parameters', 'passed', 'in', 'on', 'a', 'single', 'line', '.', 'The', 'final', 'object', 'will', 'be', 'returned', '.', 'A', 'builder', 'pattern', ',', 'on', 'the', 'other', 'hand', ',', 'is', 'in', 'essence', 'a', 'wrapper', 'object', 'around', 'all', 'the', 'possible', 'parameters', 'you', 'might', 'want', 'to', 'pass', 'into', 'a', 'constructor', 'invocation', '.', 'This', 'allows', 'you', 'to', 'use', 'setter', 'methods', 'to', 'slowly', 'build', 'up', 'your', 'parameter', 'list', '.', 'One', 'additional', 'method', 'on', 'a', 'builder', 'class', 'is', 'a', 'build', '(', ')', 'method', ',', 'which', 'simply', 'passes', 'the'

In [56]:
print(" ".join(ans_token[1]))

A factory is simply a wrapper function around a constructor ( possibly one in a different class ) . The key difference is that a factory method pattern requires the entire object to be built in a single method call , with all the parameters passed in on a single line . The final object will be returned . A builder pattern , on the other hand , is in essence a wrapper object around all the possible parameters you might want to pass into a constructor invocation . This allows you to use setter methods to slowly build up your parameter list . One additional method on a builder class is a build ( ) method , which simply passes the builder object into the desired constructor , and returns the result . In static languages like Java , this becomes more important when you have more than a handful of ( potentially optional ) parameters , as it avoids the requirement to have telescopic constructors for all the possible combinations of parameters . Also a builder allows you to use setter methods 

In [47]:
ans_filtered = []

In [48]:
for i, tok in enumerate(ans_token):
    filter = [word for word in tok if word.lower() not in stop_words]
    ans_filtered.append(filter)

In [57]:
print(ans_filtered[1])

['factory', 'simply', 'wrapper', 'function', 'around', 'constructor', '(', 'possibly', 'one', 'different', 'class', ')', '.', 'key', 'difference', 'factory', 'method', 'pattern', 'requires', 'entire', 'object', 'built', 'single', 'method', 'call', ',', 'parameters', 'passed', 'single', 'line', '.', 'final', 'object', 'returned', '.', 'builder', 'pattern', ',', 'hand', ',', 'essence', 'wrapper', 'object', 'around', 'possible', 'parameters', 'might', 'want', 'pass', 'constructor', 'invocation', '.', 'allows', 'use', 'setter', 'methods', 'slowly', 'build', 'parameter', 'list', '.', 'One', 'additional', 'method', 'builder', 'class', 'build', '(', ')', 'method', ',', 'simply', 'passes', 'builder', 'object', 'desired', 'constructor', ',', 'returns', 'result', '.', 'static', 'languages', 'like', 'Java', ',', 'becomes', 'important', 'handful', '(', 'potentially', 'optional', ')', 'parameters', ',', 'avoids', 'requirement', 'telescopic', 'constructors', 'possible', 'combinations', 'parameters',

In [58]:
print(" ".join(ans_filtered[1]))

factory simply wrapper function around constructor ( possibly one different class ) . key difference factory method pattern requires entire object built single method call , parameters passed single line . final object returned . builder pattern , hand , essence wrapper object around possible parameters might want pass constructor invocation . allows use setter methods slowly build parameter list . One additional method builder class build ( ) method , simply passes builder object desired constructor , returns result . static languages like Java , becomes important handful ( potentially optional ) parameters , avoids requirement telescopic constructors possible combinations parameters . Also builder allows use setter methods define read-only private fields directly modified constructor called . Basic Factory Example [ CODE ] Basic Builder Example [ CODE ] may worth comparing code samples two wikipedia pages : http : //en.wikipedia.org/wiki/Factory_method_pattern http : //en.wikipedia.o

## Perform **Lemmatization**

### Question lemmatization

In [62]:
import spacy

# Load the spaCy English model
nlp = spacy.load('en_core_web_sm')

In [65]:
question_lemma = [token.lemma_ for token in nlp(' '. join(question_filtered))]

In [72]:
print(question_lemma)
print(' '.join(question_lemma))

['difference', 'Builder', 'Design', 'pattern', 'factory', 'Design', 'pattern', '?', '[', 'close', ']']
difference Builder Design pattern factory Design pattern ? [ close ]


### Answer Lemmatization

In [63]:
ans_lemma = []

In [68]:
for i, tok in enumerate(ans_token):
    doc = nlp(' '.join(ans_filtered[i]))
    lemmatized_tokens = [token.lemma_ for token in doc]
    ans_lemma.append(lemmatized_tokens)

In [75]:
print(' '.join(ans_lemma[1]))

factory simply wrapper function around constructor ( possibly one different class ) . key difference factory method pattern require entire object build single method call , parameter pass single line . final object return . builder pattern , hand , essence wrapper object around possible parameter might want pass constructor invocation . allow use setter method slowly build parameter list . one additional method builder class build ( ) method , simply pass builder object desire constructor , return result . static language like Java , become important handful ( potentially optional ) parameter , avoid requirement telescopic constructor possible combination parameter . also builder allow use setter method define read - only private field directly modify constructor call . Basic Factory Example [ code ] Basic Builder Example [ CODE ] may worth compare code sample two wikipedia page : http : //en.wikipedia.org / wiki / factory_method_pattern http : //en.wikipedia.org / wiki / Builder_patte

## Perform **POS Tagging**

In [76]:
def pos_tagging(text):
    # Process the text using spaCy
    doc = nlp(text)

    # Extract POS tags for each token in the text
    pos_tags = [(token.text, token.pos_) for token in doc]

    return pos_tags

In [77]:
question_pos = pos_tagging(' '.join(question_lemma))

In [78]:
question_pos

[('difference', 'NOUN'),
 ('Builder', 'PROPN'),
 ('Design', 'PROPN'),
 ('pattern', 'NOUN'),
 ('factory', 'NOUN'),
 ('Design', 'NOUN'),
 ('pattern', 'NOUN'),
 ('?', 'PUNCT'),
 ('[', 'PUNCT'),
 ('close', 'ADV'),
 (']', 'PUNCT')]

In [80]:
ans_pos = []

In [81]:
for ans in ans_lemma:
    ans_pos.append(pos_tagging(' '.join(ans)))

In [83]:
ans_pos[1]

[('factory', 'NOUN'),
 ('simply', 'ADV'),
 ('wrapper', 'NOUN'),
 ('function', 'NOUN'),
 ('around', 'ADP'),
 ('constructor', 'NOUN'),
 ('(', 'PUNCT'),
 ('possibly', 'ADV'),
 ('one', 'NUM'),
 ('different', 'ADJ'),
 ('class', 'NOUN'),
 (')', 'PUNCT'),
 ('.', 'PUNCT'),
 ('key', 'ADJ'),
 ('difference', 'NOUN'),
 ('factory', 'NOUN'),
 ('method', 'NOUN'),
 ('pattern', 'NOUN'),
 ('require', 'VERB'),
 ('entire', 'ADJ'),
 ('object', 'NOUN'),
 ('build', 'VERB'),
 ('single', 'ADJ'),
 ('method', 'NOUN'),
 ('call', 'NOUN'),
 (',', 'PUNCT'),
 ('parameter', 'NOUN'),
 ('pass', 'VERB'),
 ('single', 'ADJ'),
 ('line', 'NOUN'),
 ('.', 'PUNCT'),
 ('final', 'ADJ'),
 ('object', 'NOUN'),
 ('return', 'NOUN'),
 ('.', 'PUNCT'),
 ('builder', 'NOUN'),
 ('pattern', 'NOUN'),
 (',', 'PUNCT'),
 ('hand', 'NOUN'),
 (',', 'PUNCT'),
 ('essence', 'NOUN'),
 ('wrapper', 'NOUN'),
 ('object', 'NOUN'),
 ('around', 'ADP'),
 ('possible', 'ADJ'),
 ('parameter', 'NOUN'),
 ('might', 'AUX'),
 ('want', 'VERB'),
 ('pass', 'VERB'),
 ('co

# Summary

Text Cleansing techniques used:

* Noise removal (from removing HTML tags, addressing stopwords)
* Data Making (from masking `code` environment into `[CODE]`

Text Preprocessing techniques used:

* Tokenization
* Lemmatization
* POS Tagging